# 05a — Facet similarity

Compare DayHikes on Facets. Each Facet gives a FacetScore in [0, 1] for a pair
of DayHikes: 1 alike, 0 unlike. The Hiker's FacetWeights combine them into one
score with a coverage figure. This is the notebook design phase of ADR-0011
(proposed); production Ranking stays ADR-0003 cosine.

Facts come from `{catalog}.similarity_facts.pkl`, written by
`05_similarity_facts.ipynb`. When that table is absent (the committed OSM
fixture `tests/fixtures/catalog.osm`, and always under pytest) this notebook
builds the facts itself with the same `dayhike.similarity` helpers.

**Facets here:** Effort, Altitude.

Each Facet section: the measures and their constants with the region's spread,
then verification. A verification states the expected verdict and the reason
before the score is computed, then prints a grep-able line
`verify<TAB><facet><TAB><check><TAB><score><TAB>PASS|FAIL`.

In [ ]:
import os
import tempfile
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd

from dayhike import build_catalog, geometry_store_path_for
from dayhike.similarity import (
    altitude_facts,
    combined_score,
    effort_facts,
    facet_score,
    load_region,
    ratio,
    similarity_facts_path_for,
    tolerance,
)

if "PYTEST_CURRENT_TEST" in os.environ:
    import matplotlib as mpl

    mpl.use("Agg")
import matplotlib.pyplot as plt


def show_fig(fig):
    try:
        from IPython.display import display

        display(fig)
    except ImportError:
        pass
    plt.close(fig)


repo = Path.cwd().resolve()
for candidate in (repo, *repo.parents):
    fixture = candidate / "tests" / "fixtures" / "catalog.osm"
    if fixture.is_file():
        repo = candidate
        break
else:
    raise FileNotFoundError("tests/fixtures/catalog.osm")

local = repo / "data" / "catalog.sqlite"
on_fixture = not (
    local.is_file()
    and geometry_store_path_for(local).is_file()
    and "PYTEST_CURRENT_TEST" not in os.environ
)
if on_fixture:
    catalog_path = Path(tempfile.mkdtemp()) / "catalog.sqlite"
    build_catalog(fixture, catalog_path)
else:
    catalog_path = local
print("catalog", catalog_path)

## Load the facts

Read the facts table when it exists. Otherwise load the region (lat 42.3–46.9,
lon 13.3–19.5) and build the same table as `05`.

The reversed-direction checks need vertex chains, which the table does not
hold. With the table present, `vertices` and `dayhikes` are a small Julian Alps
box (lat 46.3–46.5, lon 13.6–14.0) used only for those checks; without it they
are the whole region.

In [ ]:
REGION_LAT = (42.3, 46.9)
REGION_LON = (13.3, 19.5)
CHECK_LAT = (46.3, 46.5)
CHECK_LON = (13.6, 14.0)

facts_path = similarity_facts_path_for(catalog_path)
if not on_fixture and facts_path.is_file():
    facts = pd.read_pickle(facts_path)
    vertices, _, dayhikes, _ = load_region(
        catalog_path, lat_range=CHECK_LAT, lon_range=CHECK_LON
    )
    print("facts read from", facts_path)
else:
    vertices, _, dayhikes, _ = load_region(
        catalog_path, lat_range=REGION_LAT, lon_range=REGION_LON
    )
    facts = gpd.GeoDataFrame(
        dayhikes.set_index("id")[
            ["name", "name_origin", "trailhead_latitude", "trailhead_longitude", "geometry"]
        ]
        .join(effort_facts(dayhikes, vertices))
        .join(altitude_facts(dayhikes)),
        geometry="geometry",
        crs=dayhikes.crs,
    )
    print("facts built from the region")
print(f"facts_rows\t{len(facts)}")
print(f"check_dayhikes\t{len(dayhikes)}")

## Measure kinds and combining

All functions are in `dayhike.similarity`. Each returns a value in [0, 1], or
None when either DayHike lacks the fact.

| Kind | Rule | Used for |
| --- | --- | --- |
| `ratio` | min(a′, b′) / max(a′, b′), a′ = max(a, floor) | sizes: 15 vs 20 km is closer than 5 vs 10 km |
| `tolerance` | max(0, 1 − abs(a − b) / D) | levels: 1800 vs 2000 m is close, 300 vs 2000 m is not |
| `mix` | Σ min(pᵢ, qᵢ) over fractions summing to 1 | LandCover, SurfaceMix (later Facets) |
| `presence` | share of yes/no flags that agree | Huts, Peaks, Viewpoints (later Facets) |

- FacetScore (`facet_score`): the plain mean of the Facet's measures that are
  not missing. A missing Difficulty drops out; length and gain still count.
- Combined score (`combined_score`): Σ wᶠ·sᶠ / Σ wᶠ over Facets with a
  FacetScore and wᶠ > 0. FacetWeights can be in any scale (50/30/20 or 5/3/2);
  weight 0 switches a Facet off. Coverage is the scored weight over all positive
  weight. No scored Facet gives a missing score.
- Several Seeds (`multi_seed_score`): a candidate's best combined score to any
  one Seed, never a mix of Seeds.

`measure_spread` prints a measure's spread in the region and, for a fixed
sample of random pairs, the median score its constant gives.

In [ ]:
rng = np.random.default_rng(0)
left_ids = rng.choice(facts.index.to_numpy(), 2000)
right_ids = rng.choice(facts.index.to_numpy(), 2000)


def measure_spread(column, measure, constant):
    values = facts[column]
    q = values.quantile([0.05, 0.25, 0.5, 0.75, 0.95]).round(1).tolist()
    pair_scores = [
        measure(facts.at[a, column], facts.at[b, column])
        for a, b in zip(left_ids, right_ids)
    ]
    pair_scores = [s for s in pair_scores if s is not None]
    median = float(np.median(pair_scores)) if pair_scores else float("nan")
    print(
        f"measure\t{column}\t{constant}\tmissing={values.isna().mean():.3f}\t"
        f"q05,q25,q50,q75,q95={q}\trandom_pair_median={median:.2f}"
    )

### Pair verification helpers

- `pair_rows` looks up the named DayHikes (each must be a unique `osm_name`) in
  the order similar A, similar B, dissimilar A, dissimilar B.
- `show_pairs` prints their facts and draws each DayHike line on a small
  topographic map (the line alone when offline).
- `verify_pairs` scores both pairs. Both `verify` lines PASS when the similar
  pair scores higher than the dissimilar one. The soft targets (similar ≥ 0.7,
  dissimilar ≤ 0.4) are printed on separate `soft` lines.

In [ ]:
def pair_rows(similar_names, dissimilar_names):
    names = [*similar_names, *dissimilar_names]
    named = facts[(facts["name_origin"] == "osm_name") & facts["name"].isin(names)]
    assert sorted(named["name"]) == sorted(set(names)), "pair names must be unique osm_name DayHikes"
    return named.set_index("name").loc[names]


def show_pairs(pairs, columns, title):
    print(pairs[columns].round(1).to_string())
    fig, axes = plt.subplots(1, len(pairs), figsize=(3 * len(pairs), 3))
    for ax, (name, row) in zip(axes, pairs.iterrows()):
        one = gpd.GeoDataFrame([row], geometry="geometry", crs=facts.crs).to_crs(epsg=3857)
        one.plot(ax=ax, color="#c2410c", linewidth=2)
        try:
            import contextily as cx

            cx.add_basemap(ax, source=cx.providers.OpenTopoMap)
        except Exception as error:  # offline: the line alone is still the map
            print("basemap skipped", type(error).__name__)
        ax.set_title(name, fontsize=8)
        ax.set_axis_off()
    fig.suptitle(title)
    fig.tight_layout()
    show_fig(fig)


def verify(facet, check, score, ok):
    print(f"verify\t{facet}\t{check}\t{score:.3f}\t{'PASS' if ok else 'FAIL'}")


def verify_pairs(facet, score_fn, pairs):
    similar = score_fn(pairs.iloc[0], pairs.iloc[1])
    dissimilar = score_fn(pairs.iloc[2], pairs.iloc[3])
    ok = similar > dissimilar
    verify(facet, "similar_pair", similar, ok)
    verify(facet, "dissimilar_pair", dissimilar, ok)
    print(f"soft\t{facet}\tsimilar_pair\t{similar:.3f}\t>=0.7\t{'met' if similar >= 0.7 else 'missed'}")
    print(f"soft\t{facet}\tdissimilar_pair\t{dissimilar:.3f}\t<=0.4\t{'met' if dissimilar <= 0.4 else 'missed'}")

### Synthetic check helpers

`synthetic_checks` runs three checks for one Facet and prints one `verify`
line each:

- identity: the lowest score of a DayHike against itself, which must be 1.
- symmetry: the largest gap between A vs B and B vs A over the random pairs,
  which must be 0.
- reversed: the lowest score of a DayHike's facts against the same DayHike's
  facts rebuilt from its reversed vertex chain (`reversed_vertices`), which
  must be 1.

In [ ]:
def reversed_vertices(vertices):
    last = vertices.groupby("dayhike_id")["seq"].transform("max")
    return vertices.assign(seq=last - vertices["seq"])


def synthetic_checks(facet, score_fn, forward, backward):
    """Identity, symmetry and reversed direction for one Facet."""
    identity = [score_fn(row, row) for _, row in facts.iterrows()]
    identity = [s for s in identity if s is not None]
    worst = min(identity)
    verify(facet, "identity", worst, abs(worst - 1.0) < 1e-9)

    gaps = []
    for a, b in zip(left_ids, right_ids):
        ab, ba = score_fn(facts.loc[a], facts.loc[b]), score_fn(facts.loc[b], facts.loc[a])
        if ab is not None:
            gaps.append(abs(ab - ba))
    gap = max(gaps)
    verify(facet, "symmetry_max_gap", gap, gap < 1e-9)

    turned = [score_fn(forward.loc[i], backward.loc[i]) for i in forward.index]
    turned = [s for s in turned if s is not None]
    worst = min(turned)
    verify(facet, "reversed", worst, abs(worst - 1.0) < 1e-9)


backward_vertices = reversed_vertices(vertices)

## Effort

Three measures: length, elevation gain, Difficulty rank.

| Measure | Kind | Constant | Why (region spread, printed below) |
| --- | --- | --- | --- |
| `length_m` | ratio | floor 1 km | The Catalog length band starts at 1 km, so the floor only matters for a hand-made input. 6 km vs 18 km scores 0.33. |
| `elevation_gain_m` | ratio | floor 100 m | About 9% of region DayHikes climb under 100 m. Those are flat walks and should match each other (10 m vs 30 m scores 1), not sit at the far end of a ratio. |
| `difficulty_rank` | tolerance | D = 3 | Tagged on about half of the region, mostly SAC 1–2. One grade apart scores 0.67; T2 vs T5 scores 0. |

The data does not argue for other values, so the PRD starting constants stay.

In [ ]:
LENGTH_FLOOR_M = 1_000
GAIN_FLOOR_M = 100
DIFFICULTY_D = 3


def effort_score(a, b):
    return facet_score(
        [
            ratio(a["length_m"], b["length_m"], floor=LENGTH_FLOOR_M),
            ratio(a["elevation_gain_m"], b["elevation_gain_m"], floor=GAIN_FLOOR_M),
            tolerance(a["difficulty_rank"], b["difficulty_rank"], d=DIFFICULTY_D),
        ]
    )


measure_spread("length_m", lambda a, b: ratio(a, b, floor=LENGTH_FLOOR_M), f"ratio floor={LENGTH_FLOOR_M}")
measure_spread("elevation_gain_m", lambda a, b: ratio(a, b, floor=GAIN_FLOOR_M), f"ratio floor={GAIN_FLOOR_M}")
measure_spread("difficulty_rank", lambda a, b: tolerance(a, b, d=DIFFICULTY_D), f"tolerance D={DIFFICULTY_D}")

### Effort verification (real region)

Expected before computing:

- **Similar:** "Zagorje - Završje - Kum" (Slovenia) and "Mosor-Kosa" (Dalmatia).
  Both are 6–7 km with 1,000–1,250 m of climbing at SAC 2: a solid half-day
  up a hill. They are far apart and look nothing alike, which Effort ignores.
- **Dissimilar:** "Pot kralja Triglava" and "Log - Mangart". The first is a
  3 km, under-300 m T1 walk; the second an 11 km, 2,800 m climb at T6.

Skipped on the fixture, which has no such DayHikes.

In [ ]:
if on_fixture:
    print("effort pairs skipped on the fixture")
else:
    effort_pairs = pair_rows(
        ["Zagorje - Završje - Kum", "Mosor-Kosa"],
        ["Pot kralja Triglava", "Log - Mangart"],
    )
    show_pairs(effort_pairs, ["length_m", "elevation_gain_m", "difficulty_rank"], "Effort pairs")
    verify_pairs("effort", effort_score, effort_pairs)

### Effort synthetic checks

- identity: every DayHike against itself scores 1 (worst case printed).
- symmetry: A vs B equals B vs A on 2,000 random pairs (largest gap printed).
- reversed: Effort facts rebuilt from the reversed vertex chain score 1 against
  the stored direction (worst case printed). This is the check that keeps
  elevation gain direction-free.

In [ ]:
synthetic_checks(
    "effort",
    effort_score,
    effort_facts(dayhikes, vertices),
    effort_facts(dayhikes, backward_vertices),
)

## Altitude

Two measures: highest and lowest point.

| Measure | Kind | Constant | Why (region spread, printed below) |
| --- | --- | --- | --- |
| `highest_m` | tolerance | D = 1000 m | Region tops run from the coast to about 2,800 m; the middle half spans about 440–1,440 m. 1,800 vs 2,000 m scores 0.8; a lowland hill vs a 2,500 m ridge scores 0. |
| `lowest_m` | tolerance | D = 1000 m | Lowest points run from sea level to about 2,400 m; the middle half spans about 250–900 m. A climb from the sea vs a walk from a 1,500 m pass scores 0. |

A random pair of region DayHikes differs by about 600 m on the highest point
and about 400 m on the lowest, so D = 1000 m puts a typical random pair near
0.4 and 0.6 and keeps pairs from the same kind of terrain high. The PRD
starting constants stay.

In [ ]:
HIGHEST_D_M = 1_000
LOWEST_D_M = 1_000


def altitude_score(a, b):
    return facet_score(
        [
            tolerance(a["highest_m"], b["highest_m"], d=HIGHEST_D_M),
            tolerance(a["lowest_m"], b["lowest_m"], d=LOWEST_D_M),
        ]
    )


measure_spread("highest_m", lambda a, b: tolerance(a, b, d=HIGHEST_D_M), f"tolerance D={HIGHEST_D_M}")
measure_spread("lowest_m", lambda a, b: tolerance(a, b, d=LOWEST_D_M), f"tolerance D={LOWEST_D_M}")

### Altitude verification (real region)

Expected before computing:

- **Similar:** "Vršič - Razor" and "Kotovo sedlo - Mangart". Both start from a
  high Julian Alps road (about 1,600–1,700 m) and top out at about 2,500 m.
- **Dissimilar:** "Omiš - Kula" and "Dolič - Triglav". Omiš starts at the
  Dalmatian coast and tops out below 900 m; the Triglav summit route starts
  above 2,100 m and reaches the highest point in the region.

Skipped on the fixture.

In [ ]:
if on_fixture:
    print("altitude pairs skipped on the fixture")
else:
    altitude_pairs = pair_rows(
        ["Vršič - Razor", "Kotovo sedlo - Mangart"],
        ["Omiš - Kula", "Dolič - Triglav"],
    )
    show_pairs(altitude_pairs, ["highest_m", "lowest_m"], "Altitude pairs")
    verify_pairs("altitude", altitude_score, altitude_pairs)

### Altitude synthetic checks

Same three checks. Highest and lowest point do not depend on the walking
direction, so the reversed check confirms the facts are read the same way.

In [ ]:
synthetic_checks(
    "altitude",
    altitude_score,
    altitude_facts(dayhikes),
    altitude_facts(dayhikes),
)

## Ranking one Seed

Score every region DayHike against one Seed on each Facet, combine with
FacetWeights, and list the top 10 with the per-Facet scores, the combined score
and coverage. Coverage is below 1 only when a Facet cannot be scored for that
pair. On the fixture the Seed is "Ridge Walk".

In [ ]:
SEED = "Ridge Walk" if on_fixture else "Aljažev dom - Kredarica - Triglav"
FACET_WEIGHTS = {"effort": 60, "altitude": 40}
FACET_SCORES = {"effort": effort_score, "altitude": altitude_score}

seed_rows = facts[facts["name"] == SEED]
assert len(seed_rows) == 1, f"Seed name must be unique: {SEED}"
seed = seed_rows.iloc[0]

rows = []
for dayhike_id, candidate in facts.drop(index=seed.name).iterrows():
    scores = {facet: fn(seed, candidate) for facet, fn in FACET_SCORES.items()}
    score, coverage = combined_score(scores, FACET_WEIGHTS)
    rows.append({"id": dayhike_id, "name": candidate["name"], **scores, "score": score, "coverage": coverage})
ranking = pd.DataFrame(rows).set_index("id").sort_values("score", ascending=False)

print("seed", SEED, "weights", FACET_WEIGHTS)
print(facts.loc[[seed.name], ["length_m", "elevation_gain_m", "difficulty_rank", "highest_m", "lowest_m"]].round(1).to_string())
print(ranking.head(10).round(3).to_string())
for dayhike_id, row in ranking.head(10).iterrows():
    print(f"rank\t{row['name']}\t{row['score']:.3f}\t{row['coverage']:.2f}")